In [1]:
import pandas as pd
import joblib
import os
import numpy as np

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error


# ==============================
# CONFIG
# ==============================

DATA_PATH = "../../Datasets/O1D1/Ponds1.csv"   # Adjust if needed
SAVE_DIR = "../../ml/models/imputation"

os.makedirs(SAVE_DIR, exist_ok=True)

FEATURES = [
    "do",
    "temperature",
    "ph",
    "turbidity",
    "ammonia",
    "nitrate"
]


# ==============================
# LOAD DATA
# ==============================

df = pd.read_csv(DATA_PATH)

# Clean accidental spaces in column names
df.columns = df.columns.str.strip()

# Rename raw CSV columns → ML-friendly names
df = df.rename(columns={
    "DO": "do",
    "TEMP": "temperature",
    "PH": "ph",
    "TURBIDITY": "turbidity",
    "AMMONIA(mg/l)": "ammonia",
    "NITRATE(PPM)": "nitrate"
})

# Keep only required columns
df = df[FEATURES]

# Convert everything to numeric (force errors to NaN)
for col in df.columns:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Drop missing rows for training safety
df = df.dropna()

print("Dataset shape:", df.shape)


# ==============================
# TRAIN MODELS
# ==============================

results = {}

for target in FEATURES:

    print(f"\nTraining imputation model for: {target}")

    X_cols = [f for f in FEATURES if f != target]

    X = df[X_cols]
    y = df[target]

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=42
    )

    model = RandomForestRegressor(
        n_estimators=300,
        max_depth=None,
        random_state=42,
        n_jobs=-1
    )

    model.fit(X_train, y_train)

    preds = model.predict(X_test)

    r2 = r2_score(y_test, preds)
    mae = mean_absolute_error(y_test, preds)
    rmse = np.sqrt(mean_squared_error(y_test, preds))

    print(f"R2   : {r2:.4f}")
    print(f"MAE  : {mae:.4f}")
    print(f"RMSE : {rmse:.4f}")

    # Save model
    model_path = os.path.join(SAVE_DIR, f"impute_{target}.pkl")
    joblib.dump(model, model_path)

    results[target] = {
        "R2": r2,
        "MAE": mae,
        "RMSE": rmse
    }

print("\nAll imputation models trained and saved.")
print(results)

C:\Users\AdityaPandey\AppData\Local\Temp\ipykernel_16932\586581705.py:34: DtypeWarning: Columns (3,4,7,9) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(DATA_PATH)


Dataset shape: (74777, 6)

Training imputation model for: do
R2   : 0.3463
MAE  : 3.2995
RMSE : 4.4188

Training imputation model for: temperature
R2   : 0.2374
MAE  : 3.7734
RMSE : 5.1823

Training imputation model for: ph
R2   : 0.3442
MAE  : 0.7336
RMSE : 0.9047

Training imputation model for: turbidity
R2   : 0.2960
MAE  : 7.7077
RMSE : 9.7158

Training imputation model for: ammonia
R2   : 0.6426
MAE  : 0.0398
RMSE : 0.0966

Training imputation model for: nitrate
R2   : 0.6042
MAE  : 15.3805
RMSE : 22.2489

All imputation models trained and saved.
{'do': {'R2': 0.3462632254416639, 'MAE': 3.2994822312101704, 'RMSE': np.float64(4.418818636766006)}, 'temperature': {'R2': 0.2373854445850232, 'MAE': 3.7734215270621045, 'RMSE': np.float64(5.182250244102079)}, 'ph': {'R2': 0.34423198961583945, 'MAE': 0.7335725140790763, 'RMSE': np.float64(0.9047301536602526)}, 'turbidity': {'R2': 0.29599068561983544, 'MAE': 7.707706546688732, 'RMSE': np.float64(9.715805685682433)}, 'ammonia': {'R2': 0.642